In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("darkgrid")
from scipy import stats

In [2]:
sns.color_palette()

[(0.12156862745098039, 0.4666666666666667, 0.7058823529411765),
 (1.0, 0.4980392156862745, 0.054901960784313725),
 (0.17254901960784313, 0.6274509803921569, 0.17254901960784313),
 (0.8392156862745098, 0.15294117647058825, 0.1568627450980392),
 (0.5803921568627451, 0.403921568627451, 0.7411764705882353),
 (0.5490196078431373, 0.33725490196078434, 0.29411764705882354),
 (0.8901960784313725, 0.4666666666666667, 0.7607843137254902),
 (0.4980392156862745, 0.4980392156862745, 0.4980392156862745),
 (0.7372549019607844, 0.7411764705882353, 0.13333333333333333),
 (0.09019607843137255, 0.7450980392156863, 0.8117647058823529)]

## Interlimb data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import numpy as np

# Load the data from the CSV file
# Assuming 'inter_limb_data.csv' is in the same directory as the script.
try:
    df = pd.read_csv('data/integrated_data_gamma.csv')
except FileNotFoundError:
    print("Error: 'inter_limb_data.csv' not found. Please ensure the file is in the same directory.")
    exit()

# Ensure required columns exist
required_columns = ['Time (s)', 'o1', 'o2', 'F1', 'F2'] # Added F2
for col in required_columns:
    if col not in df.columns:
        print(f"Error: Column '{col}' not found in the CSV file. Please check your data.")
        exit()

# Sort data by 'Time (s)' to ensure correct animation order
df = df.sort_values(by='Time (s)')

# Setup the figure and two subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8)) # 1 row, 2 columns for two plots

# --- Plot 1: o1 vs o2 colored by F1 ---
ax1.set_title('Phase Plot: o1 vs o2 (Colored by F1)')
ax1.set_xlabel('o1')
ax1.set_ylabel('o2')
ax1.grid(True, linestyle='--', alpha=0.7)

# Initialize the scatter plot for F1
scatter_f1 = ax1.scatter([], [], s=100, c=[], cmap='viridis', vmin=df['F1'].min(), vmax=df['F1'].max())

# Set initial limits for ax1
ax1.set_xlim(df['o1'].min() - 0.1 * abs(df['o1'].min()), df['o1'].max() + 0.1 * abs(df['o1'].max()))
ax1.set_ylim(df['o2'].min() - 0.1 * abs(df['o2'].min()), df['o2'].max() + 0.1 * abs(df['o2'].max()))

# Add a colorbar for F1
cbar_f1 = fig.colorbar(scatter_f1, ax=ax1)
cbar_f1.set_label('F1 Value')

# Text element to display the current time for F1 plot
time_text_f1 = ax1.text(0.02, 0.95, '', transform=ax1.transAxes, fontsize=12, verticalalignment='top')


# --- Plot 2: o1 vs o2 colored by F2 ---
ax2.set_title('Phase Plot: o1 vs o2 (Colored by F2)')
ax2.set_xlabel('o1')
ax2.set_ylabel('o2')
ax2.grid(True, linestyle='--', alpha=0.7)

# Initialize the scatter plot for F2
scatter_f2 = ax2.scatter([], [], s=100, c=[], cmap='plasma', vmin=df['F2'].min(), vmax=df['F2'].max()) # Using 'plasma' colormap for F2

# Set initial limits for ax2 (same as ax1 for consistent scaling)
ax2.set_xlim(df['o1'].min() - 0.1 * abs(df['o1'].min()), df['o1'].max() + 0.1 * abs(df['o1'].max()))
ax2.set_ylim(df['o2'].min() - 0.1 * abs(df['o2'].min()), df['o2'].max() + 0.1 * abs(df['o2'].max()))

# Add a colorbar for F2
cbar_f2 = fig.colorbar(scatter_f2, ax=ax2)
cbar_f2.set_label('F2 Value')

# Text element to display the current time for F2 plot
time_text_f2 = ax2.text(0.02, 0.95, '', transform=ax2.transAxes, fontsize=12, verticalalignment='top')


# Animation function: This is called sequentially for each frame
def animate(i):
    # Get data up to the current frame 'i'
    current_data = df.iloc[:i+1]

    # --- Update Plot 1 (F1) ---
    scatter_f1.set_offsets(current_data[['o1', 'o2']].values)
    scatter_f1.set_array(current_data['F1'].values)

    # Make the last point larger to highlight it for F1 plot
    sizes_f1 = np.ones(len(current_data)) * 100
    if len(current_data) > 0:
        sizes_f1[-1] = 200
    scatter_f1.set_sizes(sizes_f1)

    # Update time text for F1 plot
    if not current_data.empty:
        time_text_f1.set_text(f'Time: {current_data["Time (s)"].iloc[-1]:.2f} s')

    # --- Update Plot 2 (F2) ---
    scatter_f2.set_offsets(current_data[['o1', 'o2']].values)
    scatter_f2.set_array(current_data['F2'].values)

    # Make the last point larger to highlight it for F2 plot
    sizes_f2 = np.ones(len(current_data)) * 100
    if len(current_data) > 0:
        sizes_f2[-1] = 200
    scatter_f2.set_sizes(sizes_f2)

    # Update time text for F2 plot
    if not current_data.empty:
        time_text_f2.set_text(f'Time: {current_data["Time (s)"].iloc[-1]:.2f} s')

    return scatter_f1, time_text_f1, scatter_f2, time_text_f2 # Return all updated artists

# Create the animation
# Assign the animation object to a variable (ani) to prevent it from being garbage collected.
# This ensures that the animation is retained in memory until plt.show()
# finishes or the script terminates.
ani = animation.FuncAnimation(fig, animate, frames=len(df), interval=50, blit=False)

# To save the animation as a GIF or MP4 (requires ffmpeg or imagemagick)
# Uncomment the following lines if you want to save the animation.
# E.g., ani.save('dual_phase_plot_animation.gif', writer='pillow', fps=20)
ani.save('dual_phase_plot_animation.mp4', writer='ffmpeg', fps=20)

plt.tight_layout() # Adjust layout to prevent labels from overlapping
plt.show() # Display the plot

Error: Column 'Time (s)' not found in the CSV file. Please check your data.
Error: Column 'o1' not found in the CSV file. Please check your data.
Error: Column 'o2' not found in the CSV file. Please check your data.
Error: Column 'F1' not found in the CSV file. Please check your data.
Error: Column 'F2' not found in the CSV file. Please check your data.


KeyError: 'Time (s)'

: 

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import numpy as np

# Load the data from the CSV file
# Assuming 'inter_limb_data.csv' is in the same directory as the script.
try:
    df = pd.read_csv('data/inter_limb_data.csv')
except FileNotFoundError:
    print("Error: 'inter_limb_data.csv' not found. Please ensure the file is in the same directory.")
    exit()

# Ensure required columns exist
required_columns = ['Time (s)', 'o1', 'o2', 'F1', 'F2']
for col in required_columns:
    if col not in df.columns:
        print(f"Error: Column '{col}' not found in the CSV file. Please check your data.")
        exit()

# Sort data by 'Time (s)' to ensure correct animation order
df = df.sort_values(by='Time (s)')

# Calculate overall min and max for F1 and F2 to use a common color scale
global_f_min = min(df['F1'].min(), df['F2'].min())
global_f_max = max(df['F1'].max(), df['F2'].max())

# Setup the figure and two subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8)) # 1 row, 2 columns for two plots

# --- Plot 1: o1 vs o2 colored by F1 ---
ax1.set_title('Phase Plot: o1 vs o2 (Colored by F1)')
ax1.set_xlabel('o1')
ax1.set_ylabel('o2')
ax1.grid(True, linestyle='--', alpha=0.7)

# Initialize the scatter plot for F1 with the common color scale
scatter_f1 = ax1.scatter([], [], s=100, c=[], cmap='viridis', vmin=global_f_min, vmax=global_f_max)

# Set initial limits for ax1
ax1.set_xlim(df['o1'].min() - 0.1 * abs(df['o1'].min()), df['o1'].max() + 0.1 * abs(df['o1'].max()))
ax1.set_ylim(df['o2'].min() - 0.1 * abs(df['o2'].min()), df['o2'].max() + 0.1 * abs(df['o2'].max()))

# Add a colorbar for F1
cbar_f1 = fig.colorbar(scatter_f1, ax=ax1)
cbar_f1.set_label('F1 Value')

# Text element to display the current time for F1 plot
time_text_f1 = ax1.text(0.02, 0.95, '', transform=ax1.transAxes, fontsize=12, verticalalignment='top')


# --- Plot 2: o1 vs o2 colored by F2 ---
ax2.set_title('Phase Plot: o1 vs o2 (Colored by F2)')
ax2.set_xlabel('o1')
ax2.set_ylabel('o2')
ax2.grid(True, linestyle='--', alpha=0.7)

# Initialize the scatter plot for F2 with the common color scale and 'viridis' colormap
scatter_f2 = ax2.scatter([], [], s=100, c=[], cmap='viridis', vmin=global_f_min, vmax=global_f_max) # Changed colormap to 'viridis'

# Set initial limits for ax2 (same as ax1 for consistent scaling)
ax2.set_xlim(df['o1'].min() - 0.1 * abs(df['o1'].min()), df['o1'].max() + 0.1 * abs(df['o1'].max()))
ax2.set_ylim(df['o2'].min() - 0.1 * abs(df['o2'].min()), df['o2'].max() + 0.1 * abs(df['o2'].max()))

# Add a colorbar for F2
cbar_f2 = fig.colorbar(scatter_f2, ax=ax2)
cbar_f2.set_label('F2 Value')

# Text element to display the current time for F2 plot
time_text_f2 = ax2.text(0.02, 0.95, '', transform=ax2.transAxes, fontsize=12, verticalalignment='top')


# Animation function: This is called sequentially for each frame
def animate(i):
    # Get data up to the current frame 'i'
    current_data = df.iloc[:i+1]

    # --- Update Plot 1 (F1) ---
    scatter_f1.set_offsets(current_data[['o1', 'o2']].values)
    scatter_f1.set_array(current_data['F1'].values)

    # Make the last point larger to highlight it for F1 plot
    sizes_f1 = np.ones(len(current_data)) * 100
    if len(current_data) > 0:
        sizes_f1[-1] = 200
    scatter_f1.set_sizes(sizes_f1)

    # Update time text for F1 plot
    if not current_data.empty:
        time_text_f1.set_text(f'Time: {current_data["Time (s)"].iloc[-1]:.2f} s')

    # --- Update Plot 2 (F2) ---
    scatter_f2.set_offsets(current_data[['o1', 'o2']].values)
    scatter_f2.set_array(current_data['F2'].values)

    # Make the last point larger to highlight it for F2 plot
    sizes_f2 = np.ones(len(current_data)) * 100
    if len(current_data) > 0:
        sizes_f2[-1] = 200
    scatter_f2.set_sizes(sizes_f2)

    # Update time text for F2 plot
    if not current_data.empty:
        time_text_f2.set_text(f'Time: {current_data["Time (s)"].iloc[-1]:.2f} s')

    return scatter_f1, time_text_f1, scatter_f2, time_text_f2 # Return all updated artists

# Create the animation
# Assign the animation object to a variable (ani) to prevent it from being garbage collected.
# This ensures that the animation is retained in memory until plt.show()
# finishes or the script terminates.
ani = animation.FuncAnimation(fig, animate, frames=len(df), interval=50, blit=False)

# To save the animation as a GIF or MP4 (requires ffmpeg or imagemagick)
# Uncomment the following lines if you want to save the animation.
# E.g., ani.save('dual_phase_plot_animation.gif', writer='pillow', fps=20)
ani.save('dual_phase_plot_animation_same_color.mp4', writer='ffmpeg', fps=20)

plt.tight_layout() # Adjust layout to prevent labels from overlapping
plt.show() # Display the plot


In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Load the data from the CSV file
# Assuming 'inter_limb_data.csv' is in the same directory as the script.
try:
    df = pd.read_csv('data/inter_limb_data.csv')
except FileNotFoundError:
    print("Error: 'inter_limb_data.csv' not found. Please ensure the file is in the same directory.")
    exit()

# Ensure required columns exist
required_columns = ['Time (s)', 'o1', 'o2', 'F1', 'F2']
for col in required_columns:
    if col not in df.columns:
        print(f"Error: Column '{col}' not found in the CSV file. Please check your data.")
        exit()

# Sort data by 'Time (s)' to ensure correct animation order
df = df.sort_values(by='Time (s)')

# Calculate overall min and max for F1 and F2 to use a common color scale
global_f_min = min(df['F1'].min(), df['F2'].min())
global_f_max = max(df['F1'].max(), df['F2'].max())

# Create subplots: 1 row, 2 columns
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=('Phase Plot: o1 vs o2 (Colored by F1)',
                                    'Phase Plot: o1 vs o2 (Colored by F2)'))

# --- Plot 1: o1 vs o2 colored by F1 ---
# Add an initial empty scatter trace for animation for F1
fig.add_trace(
    go.Scatter(
        x=[],
        y=[],
        mode='markers',
        marker=dict(
            size=10,
            color=[], # Placeholder for F1 values
            colorscale='Viridis', # Colormap for F1
            cmin=global_f_min,
            cmax=global_f_max,
            colorbar=dict(title='F1 Value', x=0.45) # Position colorbar for F1 plot
        ),
        name='F1 Colored'
    ),
    row=1, col=1
)

# --- Plot 2: o1 vs o2 colored by F2 ---
# Add an initial empty scatter trace for animation for F2
fig.add_trace(
    go.Scatter(
        x=[],
        y=[],
        mode='markers',
        marker=dict(
            size=10,
            color=[], # Placeholder for F2 values
            colorscale='Viridis', # Same colormap for F2
            cmin=global_f_min,
            cmax=global_f_max,
            colorbar=dict(title='F2 Value', x=1.0) # Position colorbar for F2 plot
        ),
        name='F2 Colored'
    ),
    row=1, col=2
)

# Define layout common to both subplots
fig.update_layout(
    title_text='Interactive Dual Phase Plot Animation',
    height=600,
    showlegend=False, # Hide legend as colorbars indicate meaning
    hovermode="closest", # Show hover information for the closest point
    updatemenus=[{
        'buttons': [
            {
                'args': [None, {'frame': {'duration': 50, 'redraw': False}, 'fromcurrent': True}],
                'label': 'Play',
                'method': 'animate'
            },
            {
                'args': [[None], {'frame': {'duration': 0, 'redraw': False}, 'mode': 'immediate', 'transition': {'duration': 0}}],
                'label': 'Pause',
                'method': 'animate'
            }
        ],
        'direction': 'left',
        'pad': {'r': 10, 't': 87},
        'showactive': False,
        'type': 'buttons',
        'x': 0.1,
        'xanchor': 'right',
        'y': 0,
        'yanchor': 'top'
    }]
)

# Set axes titles and ranges for both subplots
# Use consistent ranges for x and y axes for better comparison
x_range = [df['o1'].min() - 0.1 * abs(df['o1'].min()), df['o1'].max() + 0.1 * abs(df['o1'].max())]
y_range = [df['o2'].min() - 0.1 * abs(df['o2'].min()), df['o2'].max() + 0.1 * abs(df['o2'].max())]

fig.update_xaxes(title_text="o1", range=x_range, row=1, col=1)
fig.update_yaxes(title_text="o2", range=y_range, row=1, col=1)
fig.update_xaxes(title_text="o1", range=x_range, row=1, col=2)
fig.update_yaxes(title_text="o2", range=y_range, row=1, col=2)


# Create frames for the animation
frames = []
for i in range(len(df)):
    current_data = df.iloc[:i+1]
    
    # Text annotation for current time
    time_annotation = {
        'x': 0.02, 'y': 0.95, 'xref': 'paper', 'yref': 'paper',
        'text': f'Time: {current_data["Time (s)"].iloc[-1]:.2f} s',
        'showarrow': False, 'font': {'size': 12}
    }

    # Data for the first subplot (F1)
    data_f1 = go.Scatter(
        x=current_data['o1'],
        y=current_data['o2'],
        mode='markers',
        marker=dict(
            size=[10] * (len(current_data) - 1) + [20] if len(current_data) > 0 else [], # Highlight last point
            color=current_data['F1'],
            colorscale='Viridis',
            cmin=global_f_min,
            cmax=global_f_max,
        )
    )

    # Data for the second subplot (F2)
    data_f2 = go.Scatter(
        x=current_data['o1'],
        y=current_data['o2'],
        mode='markers',
        marker=dict(
            size=[10] * (len(current_data) - 1) + [20] if len(current_data) > 0 else [], # Highlight last point
            color=current_data['F2'],
            colorscale='Viridis', # Use the same colormap
            cmin=global_f_min,
            cmax=global_f_max,
        )
    )

    frames.append(
        go.Frame(
            data=[data_f1, data_f2], # Update traces for both subplots
            layout=go.Layout(annotations=[time_annotation]), # Add time annotation to each frame
            name=str(i) # Unique name for each frame
        )
    )

fig.frames = frames

# Add slider to navigate through animation frames
# This part is a bit more complex in Plotly as sliders need to reference frame names
sliders = [
    {
        'steps': [
            {
                'method': 'animate',
                'label': str(k),
                'args': [[str(k)], {'mode': 'immediate', 'frame': {'duration': 50, 'redraw': False}, 'transition': {'duration': 0}}]
            } for k in range(len(df))
        ],
        'transition': {'duration': 0},
        'x': 0.1,
        'len': 0.9,
        'currentvalue': {'font': {'size': 12}, 'prefix': 'Frame:', 'visible': True, 'xanchor': 'right'},
        'yanchor': 'top',
        'pad': {'t': 50, 'b': 10},
        'active': 0
    }
]

fig.update_layout(sliders=sliders)

# Display the interactive plot directly
fig.show()

print("Interactive phase plot animation should be displayed in a new window or inline, depending on your environment.")


In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Load the data from the CSV file
# Assuming 'inter_limb_data.csv' is in the same directory as the script.
try:
    df = pd.read_csv('data/inter_limb_data.csv')
except FileNotFoundError:
    print("Error: 'inter_limb_data.csv' not found. Please ensure the file is in the same directory.")
    exit()

# Ensure required columns exist
required_columns = ['Time (s)', 'o1', 'o2', 'F1', 'F2']
for col in required_columns:
    if col not in df.columns:
        print(f"Error: Column '{col}' not found in the CSV file. Please check your data.")
        exit()

# Sort data by 'Time (s)' to ensure correct animation order
df = df.sort_values(by='Time (s)')

# Calculate overall min and max for F1 and F2 to use a common color scale for F1/F2 plots
global_f_min = min(df['F1'].min(), df['F2'].min())
global_f_max = max(df['F1'].max(), df['F2'].max())

# Create subplots: 4 rows, 1 column for vertical arrangement
fig = make_subplots(rows=4, cols=1,
                    subplot_titles=('o1 vs Time (s)',
                                    'o2 vs Time (s)',
                                    'F1 vs Time (s)',
                                    'F2 vs Time (s)'))

# --- Plot 1: o1 vs Time (s) ---
fig.add_trace(
    go.Scatter(
        x=[],
        y=[],
        mode='lines+markers',
        name='o1',
        line=dict(color='blue', width=2),
        marker=dict(size=8, color='blue')
    ),
    row=1, col=1
)

# --- Plot 2: o2 vs Time (s) ---
fig.add_trace(
    go.Scatter(
        x=[],
        y=[],
        mode='lines+markers',
        name='o2',
        line=dict(color='red', width=2),
        marker=dict(size=8, color='red')
    ),
    row=2, col=1
)

# --- Plot 3: F1 vs Time (s) colored by F1 value ---
fig.add_trace(
    go.Scatter(
        x=[],
        y=[],
        mode='markers', # Use markers for coloring by F1 value
        name='F1',
        marker=dict(
            size=10,
            color=[], # Placeholder for F1 values
            colorscale='Viridis', # Colormap
            cmin=global_f_min,
            cmax=global_f_max,
            colorbar=dict(title='F1 Value', x=1.05, y=0.7) # Position colorbar for F1 plot
        )
    ),
    row=3, col=1
)

# --- Plot 4: F2 vs Time (s) colored by F2 value ---
fig.add_trace(
    go.Scatter(
        x=[],
        y=[],
        mode='markers', # Use markers for coloring by F2 value
        name='F2',
        marker=dict(
            size=10,
            color=[], # Placeholder for F2 values
            colorscale='Viridis', # Same colormap
            cmin=global_f_min,
            cmax=global_f_max,
            colorbar=dict(title='F2 Value', x=1.05, y=0.2) # Position colorbar for F2 plot
        )
    ),
    row=4, col=1
)

# Define overall layout
fig.update_layout(
    title_text='Interactive Multi-Variable Time-Series Animation',
    height=900, # Increased height for 4 subplots
    showlegend=False,
    hovermode="x unified", # Shows hover info for all traces at a given x-value
    updatemenus=[{
        'buttons': [
            {
                'args': [None, {'frame': {'duration': 50, 'redraw': False}, 'fromcurrent': True}],
                'label': 'Play',
                'method': 'animate'
            },
            {
                'args': [[None], {'frame': {'duration': 0, 'redraw': False}, 'mode': 'immediate', 'transition': {'duration': 0}}],
                'label': 'Pause',
                'method': 'animate'
            }
        ],
        'direction': 'left',
        'pad': {'r': 10, 't': 87},
        'showactive': False,
        'type': 'buttons',
        'x': 0.1,
        'xanchor': 'right',
        'y': 0,
        'yanchor': 'top'
    }]
)

# Set axes titles and ranges for all subplots
time_range = [df['Time (s)'].min() - 0.1 * abs(df['Time (s)'].min()), df['Time (s)'].max() + 0.1 * abs(df['Time (s)'].max())]

fig.update_xaxes(title_text="Time (s)", range=time_range, row=1, col=1)
fig.update_yaxes(title_text="o1", row=1, col=1)

fig.update_xaxes(title_text="Time (s)", range=time_range, row=2, col=1)
fig.update_yaxes(title_text="o2", row=2, col=1)

fig.update_xaxes(title_text="Time (s)", range=time_range, row=3, col=1)
fig.update_yaxes(title_text="F1", row=3, col=1)

fig.update_xaxes(title_text="Time (s)", range=time_range, row=4, col=1)
fig.update_yaxes(title_text="F2", row=4, col=1)


# Create frames for the animation
frames = []
for i in range(len(df)):
    current_data = df.iloc[:i+1]
    
    # Text annotation for current time, placed in the top subplot
    time_annotation = {
        'x': 0.02, 'y': 0.95, 'xref': 'paper', 'yref': 'paper',
        'text': f'Current Time: {current_data["Time (s)"].iloc[-1]:.2f} s',
        'showarrow': False, 'font': {'size': 12},
        'xanchor': 'left', 'yanchor': 'top'
    }

    # Data for o1 plot
    data_o1 = go.Scatter(
        x=current_data['Time (s)'],
        y=current_data['o1'],
        mode='lines+markers',
        line=dict(color='blue', width=2),
        marker=dict(size=[8] * (len(current_data) - 1) + [12] if len(current_data) > 0 else [], color='blue') # Highlight last point
    )

    # Data for o2 plot
    data_o2 = go.Scatter(
        x=current_data['Time (s)'],
        y=current_data['o2'],
        mode='lines+markers',
        line=dict(color='red', width=2),
        marker=dict(size=[8] * (len(current_data) - 1) + [12] if len(current_data) > 0 else [], color='red') # Highlight last point
    )

    # Data for F1 plot
    data_f1 = go.Scatter(
        x=current_data['Time (s)'],
        y=current_data['F1'],
        mode='markers',
        marker=dict(
            size=[10] * (len(current_data) - 1) + [15] if len(current_data) > 0 else [], # Highlight last point
            color=current_data['F1'],
            colorscale='Viridis',
            cmin=global_f_min,
            cmax=global_f_max,
        )
    )

    # Data for F2 plot
    data_f2 = go.Scatter(
        x=current_data['Time (s)'],
        y=current_data['F2'],
        mode='markers',
        marker=dict(
            size=[10] * (len(current_data) - 1) + [15] if len(current_data) > 0 else [], # Highlight last point
            color=current_data['F2'],
            colorscale='Viridis', # Use the same colormap
            cmin=global_f_min,
            cmax=global_f_max,
        )
    )

    frames.append(
        go.Frame(
            data=[data_o1, data_o2, data_f1, data_f2], # Update traces for all four subplots
            layout=go.Layout(annotations=[time_annotation]), # Add time annotation to each frame
            name=str(i) # Unique name for each frame
        )
    )

fig.frames = frames

# Add slider to navigate through animation frames
sliders = [
    {
        'steps': [
            {
                'method': 'animate',
                'label': str(k),
                'args': [[str(k)], {'mode': 'immediate', 'frame': {'duration': 50, 'redraw': False}, 'transition': {'duration': 0}}]
            } for k in range(len(df))
        ],
        'transition': {'duration': 0},
        'x': 0.1,
        'len': 0.9,
        'currentvalue': {'font': {'size': 12}, 'prefix': 'Frame:', 'visible': True, 'xanchor': 'right'},
        'yanchor': 'top',
        'pad': {'t': 50, 'b': 10},
        'active': 0
    }
]

fig.update_layout(sliders=sliders)

# Display the interactive plot directly
fig.show()

print("Interactive multi-variable time-series animation should be displayed in a new window or inline, depending on your environment.")


## Integrated data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import numpy as np

# Load the data from the CSV file
# Assuming 'inter_limb_data.csv' is in the same directory as the script.
file_name = 'data/integrated_data.csv'  # Updated to use the integrated data file
try:
    df = pd.read_csv(file_name)
except FileNotFoundError:
    print("Error: 'inter_limb_data.csv' not found. Please ensure the file is in the same directory.")
    exit()

# Ensure required columns exist
required_columns = ['Time (s)', 'o1', 'o2', 'F1', 'F2'] # Added F2
for col in required_columns:
    if col not in df.columns:
        print(f"Error: Column '{col}' not found in the CSV file. Please check your data.")
        exit()

# Sort data by 'Time (s)' to ensure correct animation order
df = df.sort_values(by='Time (s)')

# Setup the figure and two subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8)) # 1 row, 2 columns for two plots

# --- Plot 1: o1 vs o2 colored by F1 ---
ax1.set_title('Phase Plot: o1 vs o2 (Colored by F1)')
ax1.set_xlabel('o1')
ax1.set_ylabel('o2')
ax1.grid(True, linestyle='--', alpha=0.7)

# Initialize the scatter plot for F1
scatter_f1 = ax1.scatter([], [], s=100, c=[], cmap='viridis', vmin=df['F1'].min(), vmax=df['F1'].max())

# Set initial limits for ax1
ax1.set_xlim(df['o1'].min() - 0.1 * abs(df['o1'].min()), df['o1'].max() + 0.1 * abs(df['o1'].max()))
ax1.set_ylim(df['o2'].min() - 0.1 * abs(df['o2'].min()), df['o2'].max() + 0.1 * abs(df['o2'].max()))

# Add a colorbar for F1
cbar_f1 = fig.colorbar(scatter_f1, ax=ax1)
cbar_f1.set_label('F1 Value')

# Text element to display the current time for F1 plot
time_text_f1 = ax1.text(0.02, 0.95, '', transform=ax1.transAxes, fontsize=12, verticalalignment='top')


# --- Plot 2: o1 vs o2 colored by F2 ---
ax2.set_title('Phase Plot: o1 vs o2 (Colored by F2)')
ax2.set_xlabel('o1')
ax2.set_ylabel('o2')
ax2.grid(True, linestyle='--', alpha=0.7)

# Initialize the scatter plot for F2
scatter_f2 = ax2.scatter([], [], s=100, c=[], cmap='plasma', vmin=df['F2'].min(), vmax=df['F2'].max()) # Using 'plasma' colormap for F2

# Set initial limits for ax2 (same as ax1 for consistent scaling)
ax2.set_xlim(df['o1'].min() - 0.1 * abs(df['o1'].min()), df['o1'].max() + 0.1 * abs(df['o1'].max()))
ax2.set_ylim(df['o2'].min() - 0.1 * abs(df['o2'].min()), df['o2'].max() + 0.1 * abs(df['o2'].max()))

# Add a colorbar for F2
cbar_f2 = fig.colorbar(scatter_f2, ax=ax2)
cbar_f2.set_label('F2 Value')

# Text element to display the current time for F2 plot
time_text_f2 = ax2.text(0.02, 0.95, '', transform=ax2.transAxes, fontsize=12, verticalalignment='top')


# Animation function: This is called sequentially for each frame
def animate(i):
    # Get data up to the current frame 'i'
    current_data = df.iloc[:i+1]

    # --- Update Plot 1 (F1) ---
    scatter_f1.set_offsets(current_data[['o1', 'o2']].values)
    scatter_f1.set_array(current_data['F1'].values)

    # Make the last point larger to highlight it for F1 plot
    sizes_f1 = np.ones(len(current_data)) * 100
    if len(current_data) > 0:
        sizes_f1[-1] = 200
    scatter_f1.set_sizes(sizes_f1)

    # Update time text for F1 plot
    if not current_data.empty:
        time_text_f1.set_text(f'Time: {current_data["Time (s)"].iloc[-1]:.2f} s')

    # --- Update Plot 2 (F2) ---
    scatter_f2.set_offsets(current_data[['o1', 'o2']].values)
    scatter_f2.set_array(current_data['F2'].values)

    # Make the last point larger to highlight it for F2 plot
    sizes_f2 = np.ones(len(current_data)) * 100
    if len(current_data) > 0:
        sizes_f2[-1] = 200
    scatter_f2.set_sizes(sizes_f2)

    # Update time text for F2 plot
    if not current_data.empty:
        time_text_f2.set_text(f'Time: {current_data["Time (s)"].iloc[-1]:.2f} s')

    return scatter_f1, time_text_f1, scatter_f2, time_text_f2 # Return all updated artists

# Create the animation
# Assign the animation object to a variable (ani) to prevent it from being garbage collected.
# This ensures that the animation is retained in memory until plt.show()
# finishes or the script terminates.
ani = animation.FuncAnimation(fig, animate, frames=len(df), interval=50, blit=False)

# To save the animation as a GIF or MP4 (requires ffmpeg or imagemagick)
# Uncomment the following lines if you want to save the animation.
# E.g., ani.save('dual_phase_plot_animation.gif', writer='pillow', fps=20)
ani.save('dual_phase_plot_animation.mp4', writer='ffmpeg', fps=20)

plt.tight_layout() # Adjust layout to prevent labels from overlapping
plt.show() # Display the plot

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import numpy as np

# Load the data from the CSV file
# Assuming 'inter_limb_data.csv' is in the same directory as the script.
file_name = 'data/integrated_data.csv'  # Updated to use the integrated data file
try:
    df = pd.read_csv(file_name)
except FileNotFoundError:
    print("Error: 'inter_limb_data.csv' not found. Please ensure the file is in the same directory.")
    exit()

# Ensure required columns exist
required_columns = ['Time (s)', 'o1', 'o2', 'F1', 'F2']
for col in required_columns:
    if col not in df.columns:
        print(f"Error: Column '{col}' not found in the CSV file. Please check your data.")
        exit()

# Sort data by 'Time (s)' to ensure correct animation order
df = df.sort_values(by='Time (s)')

# Calculate overall min and max for F1 and F2 to use a common color scale
global_f_min = min(df['F1'].min(), df['F2'].min())
global_f_max = max(df['F1'].max(), df['F2'].max())

# Setup the figure and two subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8)) # 1 row, 2 columns for two plots

# --- Plot 1: o1 vs o2 colored by F1 ---
ax1.set_title('Phase Plot: o1 vs o2 (Colored by F1)')
ax1.set_xlabel('o1')
ax1.set_ylabel('o2')
ax1.grid(True, linestyle='--', alpha=0.7)

# Initialize the scatter plot for F1 with the common color scale
scatter_f1 = ax1.scatter([], [], s=100, c=[], cmap='viridis', vmin=global_f_min, vmax=global_f_max)

# Set initial limits for ax1
ax1.set_xlim(df['o1'].min() - 0.1 * abs(df['o1'].min()), df['o1'].max() + 0.1 * abs(df['o1'].max()))
ax1.set_ylim(df['o2'].min() - 0.1 * abs(df['o2'].min()), df['o2'].max() + 0.1 * abs(df['o2'].max()))

# Add a colorbar for F1
cbar_f1 = fig.colorbar(scatter_f1, ax=ax1)
cbar_f1.set_label('F1 Value')

# Text element to display the current time for F1 plot
time_text_f1 = ax1.text(0.02, 0.95, '', transform=ax1.transAxes, fontsize=12, verticalalignment='top')


# --- Plot 2: o1 vs o2 colored by F2 ---
ax2.set_title('Phase Plot: o1 vs o2 (Colored by F2)')
ax2.set_xlabel('o1')
ax2.set_ylabel('o2')
ax2.grid(True, linestyle='--', alpha=0.7)

# Initialize the scatter plot for F2 with the common color scale and 'viridis' colormap
scatter_f2 = ax2.scatter([], [], s=100, c=[], cmap='viridis', vmin=global_f_min, vmax=global_f_max) # Changed colormap to 'viridis'

# Set initial limits for ax2 (same as ax1 for consistent scaling)
ax2.set_xlim(df['o1'].min() - 0.1 * abs(df['o1'].min()), df['o1'].max() + 0.1 * abs(df['o1'].max()))
ax2.set_ylim(df['o2'].min() - 0.1 * abs(df['o2'].min()), df['o2'].max() + 0.1 * abs(df['o2'].max()))

# Add a colorbar for F2
cbar_f2 = fig.colorbar(scatter_f2, ax=ax2)
cbar_f2.set_label('F2 Value')

# Text element to display the current time for F2 plot
time_text_f2 = ax2.text(0.02, 0.95, '', transform=ax2.transAxes, fontsize=12, verticalalignment='top')


# Animation function: This is called sequentially for each frame
def animate(i):
    # Get data up to the current frame 'i'
    current_data = df.iloc[:i+1]

    # --- Update Plot 1 (F1) ---
    scatter_f1.set_offsets(current_data[['o1', 'o2']].values)
    scatter_f1.set_array(current_data['F1'].values)

    # Make the last point larger to highlight it for F1 plot
    sizes_f1 = np.ones(len(current_data)) * 100
    if len(current_data) > 0:
        sizes_f1[-1] = 200
    scatter_f1.set_sizes(sizes_f1)

    # Update time text for F1 plot
    if not current_data.empty:
        time_text_f1.set_text(f'Time: {current_data["Time (s)"].iloc[-1]:.2f} s')

    # --- Update Plot 2 (F2) ---
    scatter_f2.set_offsets(current_data[['o1', 'o2']].values)
    scatter_f2.set_array(current_data['F2'].values)

    # Make the last point larger to highlight it for F2 plot
    sizes_f2 = np.ones(len(current_data)) * 100
    if len(current_data) > 0:
        sizes_f2[-1] = 200
    scatter_f2.set_sizes(sizes_f2)

    # Update time text for F2 plot
    if not current_data.empty:
        time_text_f2.set_text(f'Time: {current_data["Time (s)"].iloc[-1]:.2f} s')

    return scatter_f1, time_text_f1, scatter_f2, time_text_f2 # Return all updated artists

# Create the animation
# Assign the animation object to a variable (ani) to prevent it from being garbage collected.
# This ensures that the animation is retained in memory until plt.show()
# finishes or the script terminates.
ani = animation.FuncAnimation(fig, animate, frames=len(df), interval=50, blit=False)

# To save the animation as a GIF or MP4 (requires ffmpeg or imagemagick)
# Uncomment the following lines if you want to save the animation.
# E.g., ani.save('dual_phase_plot_animation.gif', writer='pillow', fps=20)
ani.save('dual_phase_plot_animation_same_color.mp4', writer='ffmpeg', fps=20)

plt.tight_layout() # Adjust layout to prevent labels from overlapping
plt.show() # Display the plot


In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Load the data from the CSV file
# Assuming 'inter_limb_data.csv' is in the same directory as the script.
file_name = 'data/integrated_data.csv'  # Updated to use the integrated data file
try:
    df = pd.read_csv(file_name)
except FileNotFoundError:
    print("Error: 'inter_limb_data.csv' not found. Please ensure the file is in the same directory.")
    exit()

# Ensure required columns exist
required_columns = ['Time (s)', 'o1', 'o2', 'F1', 'F2']
for col in required_columns:
    if col not in df.columns:
        print(f"Error: Column '{col}' not found in the CSV file. Please check your data.")
        exit()

# Sort data by 'Time (s)' to ensure correct animation order
df = df.sort_values(by='Time (s)')

# Calculate overall min and max for F1 and F2 to use a common color scale
global_f_min = min(df['F1'].min(), df['F2'].min())
global_f_max = max(df['F1'].max(), df['F2'].max())

# Create subplots: 1 row, 2 columns
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=('Phase Plot: o1 vs o2 (Colored by F1)',
                                    'Phase Plot: o1 vs o2 (Colored by F2)'))

# --- Plot 1: o1 vs o2 colored by F1 ---
# Add an initial empty scatter trace for animation for F1
fig.add_trace(
    go.Scatter(
        x=[],
        y=[],
        mode='markers',
        marker=dict(
            size=10,
            color=[], # Placeholder for F1 values
            colorscale='Viridis', # Colormap for F1
            cmin=global_f_min,
            cmax=global_f_max,
            colorbar=dict(title='F1 Value', x=0.45) # Position colorbar for F1 plot
        ),
        name='F1 Colored'
    ),
    row=1, col=1
)

# --- Plot 2: o1 vs o2 colored by F2 ---
# Add an initial empty scatter trace for animation for F2
fig.add_trace(
    go.Scatter(
        x=[],
        y=[],
        mode='markers',
        marker=dict(
            size=10,
            color=[], # Placeholder for F2 values
            colorscale='Viridis', # Same colormap for F2
            cmin=global_f_min,
            cmax=global_f_max,
            colorbar=dict(title='F2 Value', x=1.0) # Position colorbar for F2 plot
        ),
        name='F2 Colored'
    ),
    row=1, col=2
)

# Define layout common to both subplots
fig.update_layout(
    title_text='Interactive Dual Phase Plot Animation',
    height=600,
    showlegend=False, # Hide legend as colorbars indicate meaning
    hovermode="closest", # Show hover information for the closest point
    updatemenus=[{
        'buttons': [
            {
                'args': [None, {'frame': {'duration': 50, 'redraw': False}, 'fromcurrent': True}],
                'label': 'Play',
                'method': 'animate'
            },
            {
                'args': [[None], {'frame': {'duration': 0, 'redraw': False}, 'mode': 'immediate', 'transition': {'duration': 0}}],
                'label': 'Pause',
                'method': 'animate'
            }
        ],
        'direction': 'left',
        'pad': {'r': 10, 't': 87},
        'showactive': False,
        'type': 'buttons',
        'x': 0.1,
        'xanchor': 'right',
        'y': 0,
        'yanchor': 'top'
    }]
)

# Set axes titles and ranges for both subplots
# Use consistent ranges for x and y axes for better comparison
x_range = [df['o1'].min() - 0.1 * abs(df['o1'].min()), df['o1'].max() + 0.1 * abs(df['o1'].max())]
y_range = [df['o2'].min() - 0.1 * abs(df['o2'].min()), df['o2'].max() + 0.1 * abs(df['o2'].max())]

fig.update_xaxes(title_text="o1", range=x_range, row=1, col=1)
fig.update_yaxes(title_text="o2", range=y_range, row=1, col=1)
fig.update_xaxes(title_text="o1", range=x_range, row=1, col=2)
fig.update_yaxes(title_text="o2", range=y_range, row=1, col=2)


# Create frames for the animation
frames = []
for i in range(len(df)):
    current_data = df.iloc[:i+1]
    
    # Text annotation for current time
    time_annotation = {
        'x': 0.02, 'y': 0.95, 'xref': 'paper', 'yref': 'paper',
        'text': f'Time: {current_data["Time (s)"].iloc[-1]:.2f} s',
        'showarrow': False, 'font': {'size': 12}
    }

    # Data for the first subplot (F1)
    data_f1 = go.Scatter(
        x=current_data['o1'],
        y=current_data['o2'],
        mode='markers',
        marker=dict(
            size=[10] * (len(current_data) - 1) + [20] if len(current_data) > 0 else [], # Highlight last point
            color=current_data['F1'],
            colorscale='Viridis',
            cmin=global_f_min,
            cmax=global_f_max,
        )
    )

    # Data for the second subplot (F2)
    data_f2 = go.Scatter(
        x=current_data['o1'],
        y=current_data['o2'],
        mode='markers',
        marker=dict(
            size=[10] * (len(current_data) - 1) + [20] if len(current_data) > 0 else [], # Highlight last point
            color=current_data['F2'],
            colorscale='Viridis', # Use the same colormap
            cmin=global_f_min,
            cmax=global_f_max,
        )
    )

    frames.append(
        go.Frame(
            data=[data_f1, data_f2], # Update traces for both subplots
            layout=go.Layout(annotations=[time_annotation]), # Add time annotation to each frame
            name=str(i) # Unique name for each frame
        )
    )

fig.frames = frames

# Add slider to navigate through animation frames
# This part is a bit more complex in Plotly as sliders need to reference frame names
sliders = [
    {
        'steps': [
            {
                'method': 'animate',
                'label': str(k),
                'args': [[str(k)], {'mode': 'immediate', 'frame': {'duration': 50, 'redraw': False}, 'transition': {'duration': 0}}]
            } for k in range(len(df))
        ],
        'transition': {'duration': 0},
        'x': 0.1,
        'len': 0.9,
        'currentvalue': {'font': {'size': 12}, 'prefix': 'Frame:', 'visible': True, 'xanchor': 'right'},
        'yanchor': 'top',
        'pad': {'t': 50, 'b': 10},
        'active': 0
    }
]

fig.update_layout(sliders=sliders)

# Display the interactive plot directly
fig.show()

print("Interactive phase plot animation should be displayed in a new window or inline, depending on your environment.")


In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Load the data from the CSV file
# Assuming 'inter_limb_data.csv' is in the same directory as the script.
file_name = 'data/integrated_data.csv'  # Updated to use the integrated data file
try:
    df = pd.read_csv(file_name)
except FileNotFoundError:
    print("Error: 'inter_limb_data.csv' not found. Please ensure the file is in the same directory.")
    exit()

# Ensure required columns exist
required_columns = ['Time (s)', 'o1', 'o2', 'F1', 'F2']
for col in required_columns:
    if col not in df.columns:
        print(f"Error: Column '{col}' not found in the CSV file. Please check your data.")
        exit()

# Sort data by 'Time (s)' to ensure correct animation order
df = df.sort_values(by='Time (s)')

# Calculate overall min and max for F1 and F2 to use a common color scale for F1/F2 plots
global_f_min = min(df['F1'].min(), df['F2'].min())
global_f_max = max(df['F1'].max(), df['F2'].max())

# Create subplots: 4 rows, 1 column for vertical arrangement
fig = make_subplots(rows=4, cols=1,
                    subplot_titles=('o1 vs Time (s)',
                                    'o2 vs Time (s)',
                                    'F1 vs Time (s)',
                                    'F2 vs Time (s)'))

# --- Plot 1: o1 vs Time (s) ---
fig.add_trace(
    go.Scatter(
        x=[],
        y=[],
        mode='lines+markers',
        name='o1',
        line=dict(color='blue', width=2),
        marker=dict(size=8, color='blue')
    ),
    row=1, col=1
)

# --- Plot 2: o2 vs Time (s) ---
fig.add_trace(
    go.Scatter(
        x=[],
        y=[],
        mode='lines+markers',
        name='o2',
        line=dict(color='red', width=2),
        marker=dict(size=8, color='red')
    ),
    row=2, col=1
)

# --- Plot 3: F1 vs Time (s) colored by F1 value ---
fig.add_trace(
    go.Scatter(
        x=[],
        y=[],
        mode='markers', # Use markers for coloring by F1 value
        name='F1',
        marker=dict(
            size=10,
            color=[], # Placeholder for F1 values
            colorscale='Viridis', # Colormap
            cmin=global_f_min,
            cmax=global_f_max,
            colorbar=dict(title='F1 Value', x=1.05, y=0.7) # Position colorbar for F1 plot
        )
    ),
    row=3, col=1
)

# --- Plot 4: F2 vs Time (s) colored by F2 value ---
fig.add_trace(
    go.Scatter(
        x=[],
        y=[],
        mode='markers', # Use markers for coloring by F2 value
        name='F2',
        marker=dict(
            size=10,
            color=[], # Placeholder for F2 values
            colorscale='Viridis', # Same colormap
            cmin=global_f_min,
            cmax=global_f_max,
            colorbar=dict(title='F2 Value', x=1.05, y=0.2) # Position colorbar for F2 plot
        )
    ),
    row=4, col=1
)

# Define overall layout
fig.update_layout(
    title_text='Interactive Multi-Variable Time-Series Animation',
    height=900, # Increased height for 4 subplots
    showlegend=False,
    hovermode="x unified", # Shows hover info for all traces at a given x-value
    updatemenus=[{
        'buttons': [
            {
                'args': [None, {'frame': {'duration': 50, 'redraw': False}, 'fromcurrent': True}],
                'label': 'Play',
                'method': 'animate'
            },
            {
                'args': [[None], {'frame': {'duration': 0, 'redraw': False}, 'mode': 'immediate', 'transition': {'duration': 0}}],
                'label': 'Pause',
                'method': 'animate'
            }
        ],
        'direction': 'left',
        'pad': {'r': 10, 't': 87},
        'showactive': False,
        'type': 'buttons',
        'x': 0.1,
        'xanchor': 'right',
        'y': 0,
        'yanchor': 'top'
    }]
)

# Set axes titles and ranges for all subplots
time_range = [df['Time (s)'].min() - 0.1 * abs(df['Time (s)'].min()), df['Time (s)'].max() + 0.1 * abs(df['Time (s)'].max())]

fig.update_xaxes(title_text="Time (s)", range=time_range, row=1, col=1)
fig.update_yaxes(title_text="o1", row=1, col=1)

fig.update_xaxes(title_text="Time (s)", range=time_range, row=2, col=1)
fig.update_yaxes(title_text="o2", row=2, col=1)

fig.update_xaxes(title_text="Time (s)", range=time_range, row=3, col=1)
fig.update_yaxes(title_text="F1", row=3, col=1)

fig.update_xaxes(title_text="Time (s)", range=time_range, row=4, col=1)
fig.update_yaxes(title_text="F2", row=4, col=1)


# Create frames for the animation
frames = []
for i in range(len(df)):
    current_data = df.iloc[:i+1]
    
    # Text annotation for current time, placed in the top subplot
    time_annotation = {
        'x': 0.02, 'y': 0.95, 'xref': 'paper', 'yref': 'paper',
        'text': f'Current Time: {current_data["Time (s)"].iloc[-1]:.2f} s',
        'showarrow': False, 'font': {'size': 12},
        'xanchor': 'left', 'yanchor': 'top'
    }

    # Data for o1 plot
    data_o1 = go.Scatter(
        x=current_data['Time (s)'],
        y=current_data['o1'],
        mode='lines+markers',
        line=dict(color='blue', width=2),
        marker=dict(size=[8] * (len(current_data) - 1) + [12] if len(current_data) > 0 else [], color='blue') # Highlight last point
    )

    # Data for o2 plot
    data_o2 = go.Scatter(
        x=current_data['Time (s)'],
        y=current_data['o2'],
        mode='lines+markers',
        line=dict(color='red', width=2),
        marker=dict(size=[8] * (len(current_data) - 1) + [12] if len(current_data) > 0 else [], color='red') # Highlight last point
    )

    # Data for F1 plot
    data_f1 = go.Scatter(
        x=current_data['Time (s)'],
        y=current_data['F1'],
        mode='markers',
        marker=dict(
            size=[10] * (len(current_data) - 1) + [15] if len(current_data) > 0 else [], # Highlight last point
            color=current_data['F1'],
            colorscale='Viridis',
            cmin=global_f_min,
            cmax=global_f_max,
        )
    )

    # Data for F2 plot
    data_f2 = go.Scatter(
        x=current_data['Time (s)'],
        y=current_data['F2'],
        mode='markers',
        marker=dict(
            size=[10] * (len(current_data) - 1) + [15] if len(current_data) > 0 else [], # Highlight last point
            color=current_data['F2'],
            colorscale='Viridis', # Use the same colormap
            cmin=global_f_min,
            cmax=global_f_max,
        )
    )

    frames.append(
        go.Frame(
            data=[data_o1, data_o2, data_f1, data_f2], # Update traces for all four subplots
            layout=go.Layout(annotations=[time_annotation]), # Add time annotation to each frame
            name=str(i) # Unique name for each frame
        )
    )

fig.frames = frames

# Add slider to navigate through animation frames
sliders = [
    {
        'steps': [
            {
                'method': 'animate',
                'label': str(k),
                'args': [[str(k)], {'mode': 'immediate', 'frame': {'duration': 50, 'redraw': False}, 'transition': {'duration': 0}}]
            } for k in range(len(df))
        ],
        'transition': {'duration': 0},
        'x': 0.1,
        'len': 0.9,
        'currentvalue': {'font': {'size': 12}, 'prefix': 'Frame:', 'visible': True, 'xanchor': 'right'},
        'yanchor': 'top',
        'pad': {'t': 50, 'b': 10},
        'active': 0
    }
]

fig.update_layout(sliders=sliders)

# Display the interactive plot directly
fig.show()

print("Interactive multi-variable time-series animation should be displayed in a new window or inline, depending on your environment.")


## Analyse Gait from learned weight of intralimb mechanism

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Load the data from the CSV file
# Ensure the file path is correct: 'data/learned_intralimb_gait_data.csv'
try:
    df = pd.read_csv('data/learned_intralimb_gait_data.csv') # Updated file path
except FileNotFoundError:
    print("Error: 'data/learned_intralimb_gait_data.csv' not found. Please ensure the file is in the 'data/' directory.")
    exit()

# Define column names based on the provided format
time_col = 'Time (s)'
o_vars = ['o1', 'o2']
fc_exfc_vars = ['FC_R1', 'FC_R2', 'FC_L1', 'FC_L2', 'ExFC_R1', 'ExFC_R2', 'ExFC_L1', 'ExFC_L2']
all_plot_vars = o_vars + fc_exfc_vars # List of all variables to plot

# Ensure all required columns exist in the DataFrame
required_columns = [time_col] + all_plot_vars
for col in required_columns:
    if col not in df.columns:
        print(f"Error: Column '{col}' not found in the CSV file. Please check your data headers.")
        exit()

# Sort data by 'Time (s)' to ensure correct animation order
df = df.sort_values(by=time_col)

# Define the threshold for highlighting
HIGHLIGHT_THRESHOLD = 0.7

# Determine the number of subplots (1 for o1/o2, and 1 for each FC/ExFC variable)
num_subplots = 1 + len(fc_exfc_vars)
subplot_titles = [f'{o_vars[0]} & {o_vars[1]} vs {time_col}'] + [f'{var} vs {time_col}' for var in fc_exfc_vars]

# Create subplots: num_subplots rows, 1 column for vertical arrangement
# Adjusted vertical_spacing for narrower space between subplots
fig = make_subplots(rows=num_subplots, cols=1,
                    subplot_titles=subplot_titles,
                    vertical_spacing=0.03) # Adjusted vertical spacing

# --- Initial Traces for each subplot ---
# Subplot 1: o1 and o2 vs Time (s)
fig.add_trace(
    go.Scatter(
        x=[], y=[], mode='lines+markers', name=o_vars[0],
        line=dict(color='blue', width=2),
        marker=dict(size=5, color='blue'), # Smaller initial marker size
        # Correctly escape curly braces for Plotly's hovertemplate within the f-string
        hovertemplate=f'Time: %{{x:.3f}}s<br>{o_vars[0]}: %{{y:.3f}}<extra></extra>'
    ),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(
        x=[], y=[], mode='lines+markers', name=o_vars[1],
        line=dict(color='red', width=2),
        marker=dict(size=5, color='red'), # Smaller initial marker size
        # Correctly escape curly braces for Plotly's hovertemplate within the f-string
        hovertemplate=f'Time: %{{x:.3f}}s<br>{o_vars[1]}: %{{y:.3f}}<extra></extra>'
    ),
    row=1, col=1
)

# Subsequent subplots for FC/ExFC variables
for i, var in enumerate(fc_exfc_vars):
    fig.add_trace(
        go.Scatter(
            x=[], y=[], mode='lines+markers', name=var,
            line=dict(color='gray', width=1), # Default line color
            marker=dict(size=6, color='blue'), # Marker color will be updated in frames
            # Correctly escape curly braces for Plotly's hovertemplate within the f-string
            hovertemplate=f'Time: %{{x:.3f}}s<br>{var}: %{{y:.3f}}<extra></extra>'
        ),
        row=i + 2, col=1 # +2 because row 1 is for o1/o2
    )

# --- Define overall layout ---
fig.update_layout(
    title_text='Interactive Multi-Variable Time-Series Animation',
    height=num_subplots * 200 + 100, # Adjust overall height based on number of subplots
    showlegend=True, # Show legend to differentiate o1/o2 lines
    hovermode="x unified", # Shows hover info for all traces at a given x-value
    margin=dict(t=80, b=20, l=60, r=60), # Adjust margins
    updatemenus=[{
        'buttons': [
            {
                'args': [None, {'frame': {'duration': 50, 'redraw': False}, 'fromcurrent': True}],
                'label': 'Play',
                'method': 'animate'
            },
            {
                'args': [[None], {'frame': {'duration': 0, 'redraw': False}, 'mode': 'immediate', 'transition': {'duration': 0}}],
                'label': 'Pause',
                'method': 'animate'
            }
        ],
        'direction': 'left',
        'pad': {'r': 10, 't': 87},
        'showactive': False,
        'type': 'buttons',
        'x': 0.1,
        'xanchor': 'right',
        'y': 0,
        'yanchor': 'top'
    }]
)

# --- Set axes titles and ranges for all subplots ---
time_range = [df[time_col].min(), df[time_col].max()] # Full time range for X-axis

# For o1 and o2 subplot
fig.update_xaxes(title_text=time_col, range=time_range, row=1, col=1)
fig.update_yaxes(title_text=f'{o_vars[0]} & {o_vars[1]}',
                 range=[min(df[o_vars[0]].min(), df[o_vars[1]].min()) * 0.9,
                        max(df[o_vars[0]].max(), df[o_vars[1]].max()) * 1.1],
                 row=1, col=1)

# For FC/ExFC variables
for i, var in enumerate(fc_exfc_vars):
    fig.update_xaxes(title_text=time_col, range=time_range, row=i + 2, col=1)
    fig.update_yaxes(title_text=var,
                     range=[df[var].min() * 0.9, df[var].max() * 1.1],
                     row=i + 2, col=1)

# --- Create frames for the animation ---
frames = []
# The first two traces are o1 and o2. Subsequent traces correspond to fc_exfc_vars.
num_initial_traces = 2

for i in range(len(df)):
    current_data = df.iloc[:i+1]
    
    # Text annotation for current time, placed in the top subplot
    time_annotation = {
        'x': 0.02, 'y': 0.95, 'xref': 'paper', 'yref': 'paper',
        'text': f'Current Time: {current_data[time_col].iloc[-1]:.3f} s',
        'showarrow': False, 'font': {'size': 14},
        'xanchor': 'left', 'yanchor': 'top'
    }

    frame_traces = []

    # Data for o1 plot (first trace in the first subplot)
    frame_traces.append(
        go.Scatter(
            x=current_data[time_col],
            y=current_data[o_vars[0]],
            mode='lines+markers',
            line=dict(color='blue', width=2),
            # Highlight last point
            marker=dict(size=[5] * (len(current_data) - 1) + [8] if len(current_data) > 0 else [], color='blue')
        )
    )

    # Data for o2 plot (second trace in the first subplot)
    frame_traces.append(
        go.Scatter(
            x=current_data[time_col],
            y=current_data[o_vars[1]],
            mode='lines+markers',
            line=dict(color='red', width=2),
            # Highlight last point
            marker=dict(size=[5] * (len(current_data) - 1) + [8] if len(current_data) > 0 else [], color='red')
        )
    )

    # Data for FC/ExFC variables (subsequent traces in their respective subplots)
    for var_idx, var in enumerate(fc_exfc_vars):
        # Determine marker colors based on the threshold
        marker_colors = ['blue'] * len(current_data) # Default color
        marker_sizes = [6] * len(current_data)      # Default size

        if len(current_data) > 0:
            for k in range(len(current_data)):
                if current_data[var].iloc[k] > HIGHLIGHT_THRESHOLD:
                    marker_colors[k] = 'orange' # Highlight color
            
            # Highlight the last point
            marker_sizes[-1] = 9
            
        frame_traces.append(
            go.Scatter(
                x=current_data[time_col],
                y=current_data[var],
                mode='lines+markers',
                line=dict(color='gray', width=1), # Line remains gray
                marker=dict(size=marker_sizes, color=marker_colors) # Dynamic marker colors and size
            )
        )

    frames.append(
        go.Frame(
            data=frame_traces, # Update traces for all subplots
            layout=go.Layout(annotations=[time_annotation]), # Add time annotation to each frame
            name=str(i) # Unique name for each frame
        )
    )

fig.frames = frames

# Add slider to navigate through animation frames
# Note: The number of steps in the slider must match the number of frames.
sliders = [
    {
        'steps': [
            {
                'method': 'animate',
                'label': str(k),
                'args': [[str(k)], {'mode': 'immediate', 'frame': {'duration': 50, 'redraw': False}, 'transition': {'duration': 0}}]
            } for k in range(len(df))
        ],
        'transition': {'duration': 0},
        'x': 0.1,
        'len': 0.9,
        'currentvalue': {'font': {'size': 12}, 'prefix': 'Frame:', 'visible': True, 'xanchor': 'right'},
        'yanchor': 'top',
        'pad': {'t': 50, 'b': 10},
        'active': 0 # Start at the first frame
    }
]

fig.update_layout(sliders=sliders)

# Display the interactive plot directly
fig.show()

print("Interactive multi-variable time-series animation should be displayed in a new window or inline, depending on your environment.")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Load the data from the CSV file
try:
    df = pd.read_csv('data/learned_intralimb_gait_data.csv')
except FileNotFoundError:
    print("Error: 'data/learned_intralimb_gait_data.csv' not found. Please ensure the file is in the 'data/' directory.")
    exit()

# Define column names
time_col = 'Time (s)'
o_vars = ['o1', 'o2']
fc_exfc_vars = ['FC_R1', 'FC_R2', 'FC_L1', 'FC_L2', 'ExFC_R1', 'ExFC_R2', 'ExFC_L1', 'ExFC_L2']
all_plot_vars = o_vars + fc_exfc_vars # List of all variables to plot

# Ensure all required columns exist in the DataFrame
required_columns = [time_col] + all_plot_vars
for col in required_columns:
    if col not in df.columns:
        print(f"Error: Column '{col}' not found in the CSV file. Please check your data headers.")
        exit()

# Sort data by 'Time (s)' to ensure correct plotting order
df = df.sort_values(by=time_col)

# Define the threshold for highlighting
HIGHLIGHT_THRESHOLD = 0.7

# Determine the number of subplots (1 for o1/o2, and 1 for each FC/ExFC variable)
num_subplots = 1 + len(fc_exfc_vars)

# Set Seaborn style for better aesthetics
sns.set_theme(style="whitegrid", palette="deep")

# Create figure and subplots
fig, axes = plt.subplots(num_subplots, 1, figsize=(12, 2.0 * num_subplots), sharex=True) # Adjusted height
fig.suptitle('Time Series of Variables', fontsize=16, y=0.98) # Adjusted title position

# Ensure axes is an array even for a single subplot
if num_subplots == 1:
    axes = [axes]

# --- Plot 1: o1 and o2 vs Time (s) ---
axes[0].set_title(f'{o_vars[0]} & {o_vars[1]} vs {time_col}', fontsize=12)
axes[0].set_ylabel(f'{o_vars[0]} & {o_vars[1]}', fontsize=10) # Label for combined plot
sns.lineplot(x=df[time_col], y=df[o_vars[0]], ax=axes[0], lw=1.5, color=sns.color_palette()[0], label=o_vars[0])
sns.lineplot(x=df[time_col], y=df[o_vars[1]], ax=axes[0], lw=1.5, color=sns.color_palette()[3], label=o_vars[1])

# Adjusted y-axis limits to add a small buffer (e.g., 5% of the range)
min_o = min(df[o_vars[0]].min(), df[o_vars[1]].min())
max_o = max(df[o_vars[0]].max(), df[o_vars[1]].max())
range_o = max_o - min_o
axes[0].set_ylim(min_o - range_o * 0.05, max_o + range_o * 0.05)
axes[0].legend(loc='upper left', fontsize=9)


# --- Plot FC/ExFC variables ---
for i, var in enumerate(fc_exfc_vars):
    current_ax = axes[i+1]
    current_ax.set_title(f'{var} vs {time_col}', fontsize=12)
    current_ax.set_ylabel(var, fontsize=10) # Label for individual variable plot

    # Determine marker colors based on the threshold
    # Initialize colors array with dtype=object to prevent string truncation
    colors = np.array(['orange'] * len(df), dtype=object)
    colors[df[var] > HIGHLIGHT_THRESHOLD] = 'blue'

    # Plot as scatter
    current_ax.scatter(df[time_col], df[var], s=20, c=colors, edgecolor='none') # Smaller marker size
    
    # Adjusted y-axis limits to add a small buffer (e.g., 5% of the range)
    min_var = df[var].min()
    max_var = df[var].max()
    range_var = max_var - min_var
    # Handle cases where min_var and max_var are the same (flat line)
    if range_var == 0:
        current_ax.set_ylim(min_var - 0.1, max_var + 0.1) # Set a fixed small buffer
    else:
        current_ax.set_ylim(min_var - range_var * 0.05, max_var + range_var * 0.05)

# Set common X-axis label for the bottom subplot
axes[-1].set_xlabel(time_col, fontsize=12)

# Adjust layout to prevent overlapping titles/labels and reduce vertical space
plt.tight_layout(rect=[0, 0, 1, 0.96], h_pad=0.2) # rect adjusts space for suptitle, h_pad for vertical spacing

plt.show()

print("Static multi-variable time-series plot should be displayed in a new window with adjusted y-axis limits.")
